## Colour CNN Example: CIFAR-10 Dataset

This example constructs a classifier for the CIFAR-10 dataset. CIFAR stands for _Canadian Institute for Advanced Research_ and the CIFAR-10 dataset contains 60,000 32 x 32 colour images in 10 classes, airplane, automobile, bird, cat, deer, dog, frog, horse, ship and truck. The dataset is built into Keras but the homepage can be found at [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html).

In [ ]:
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from tqdm.notebook import trange
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Load and Visualize the Data

In [ ]:
# Define the transform
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load the dataset into memory
trainset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Function to load data into GPU
def load_data_to_gpu(dataset):
    data_loader = DataLoader(dataset, batch_size=len(dataset), shuffle=False)
    data_iter = iter(data_loader)
    images, labels = next(data_iter)
    return images.to('cuda'), labels.to('cuda')

# Load the train and test sets to GPU
train_images, train_labels = load_data_to_gpu(trainset)
test_images, test_labels = load_data_to_gpu(testset)

# Create TensorDatasets with data on GPU
train_dataset = TensorDataset(train_images, train_labels)
test_dataset = TensorDataset(test_images, test_labels)

# Create DataLoaders with smaller batch sizes for training and testing
trainloader = DataLoader(train_dataset, batch_size=2048, shuffle=True)
testloader = DataLoader(test_dataset, batch_size=2048, shuffle=False)

Visualize the images

In [ ]:
# Define the class names
cifar_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

# Plot some examples from the training set
dataiter = iter(trainloader)
images, labels = next(dataiter)

# Normalize images to the range 0 to 1
images = images.detach().cpu() / 2 + 0.5  # Unnormalize

fig, ax = plt.subplots(2, 5, figsize=(10, 5))
for i in range(2):
    for j in range(5):
        img = images[i * 5 + j].permute(1, 2, 0)  # Convert to (H, W, C) format
        ax[i, j].axes.xaxis.set_visible(False)
        ax[i, j].axes.yaxis.set_visible(False)
        ax[i, j].imshow(img)
        cl = labels[i * 5 + j]
        ax[i, j].set_title(cifar_classes[cl])
fig.savefig('CIFAR10Examples.png', dpi=300, bbox_inches='tight')

### Build CNN Model

The CNN model is constructed using 3 Convolutional layers and 2 Max Pooling layers before finishing with a dense layer before the softmax. This is essentially the same structure as used with MNIST but the input size and number of channels has changed.

In [ ]:
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(256, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        x = x.view(-1, 256) 
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
# Create an instance of the NeuralNetwork
model = CNNModel().to(device)

In [ ]:
print(model)

In [ ]:
no_epochs = 200

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs):
    train_errors = []
    test_errors = []
    train_accuracies = []
    test_accuracies = []

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0
        correct_train = 0

        # Training
        for batch_X, batch_y in train_loader:         
            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs.squeeze(), batch_y)
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * batch_X.size(0)
            # Calculate accuracy
            predicted = torch.argmax(outputs.squeeze(), dim=1)
            targets = batch_y
            correct_train += (predicted == targets).sum().item()

        train_loss /= len(train_loader.dataset)
        train_accuracy = 100 * correct_train / len(train_loader.dataset)
        train_errors.append(train_loss)
        train_accuracies.append(train_accuracy)
        
        # Evaluation on test set
        model.eval()
        test_loss = 0.0
        correct_test = 0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X = batch_X.to(device)
                batch_y = batch_y.to(device)
                outputs = model(batch_X)
                loss = loss_fn(outputs.squeeze(), batch_y)
                test_loss += loss.item() * batch_X.size(0)
                # Calculate accuracy
                predicted = torch.argmax(outputs.squeeze(), dim=1)
                targets = batch_y
                correct_test += (predicted == targets).sum().item()

        test_loss /= len(test_loader.dataset)
        test_accuracy = 100 * correct_test / len(test_loader.dataset)
        test_errors.append(test_loss)
        test_accuracies.append(test_accuracy)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} \
            - Train loss: {train_loss:.4f}, Test loss: {test_loss:.4f}, \
            Train Acc: {train_accuracy:.2f}%, Test Acc: {test_accuracy:.2f}%")

    history = dict()
    history['train_loss'] = train_errors
    history['test_loss'] = test_errors
    history['train_acc'] = train_accuracies
    history['test_acc'] = test_accuracies
        
    return history

In [ ]:
# Define the optimizer (Adam in this case)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Define the loss function (categorical cross-entropy in this case)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
history = train_model(model, trainloader, testloader,
                      loss_fn, optimizer, no_epochs)

### Plot Validation Accuracy

In [ ]:
epochs = range(1, no_epochs + 1)

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))

val_acc_values = history['test_acc']
train_acc_values = history['train_acc']
ax.plot(epochs, val_acc_values, 'k')
ax.plot(epochs, train_acc_values, 'k--')

ax.set_xlabel('Epochs')
ax.set_ylabel('Validation Accuracy')
fig.savefig('TestCNNCIFAR10.png', dpi=300, bbox_inches='tight')